In [8]:
import torch
import numpy as np
import yaml
from tqdm import tqdm
from argparse import Namespace

import os
import sys

# Add project root to path (adjust the path as needed for your notebook location)
project_root = os.path.abspath('..')  # or '../..' if notebook is nested deeper
sys.path.insert(0, project_root)
os.chdir(project_root)  # Change working directory to project root

print(f"Working directory: {os.getcwd()}")

from models.GRASSY_model import GRASSY
from models.ScatteringTransform import GraphScatteringTransform
from datasets.load_ZINC_tranche import ZINCDataset

Working directory: /nfs/roberts/project/pi_sk2433/jcr222/workspace


In [9]:
class FixedScatteringTransform:
    """
    Transform that applies fixed GraphScatteringTransform to PyG Data objects.
    No learnable parameters - computes scattering coefficients deterministically.
    """

    def __init__(self, in_channels, J=4, num_moments=4):
        self.scattering = GraphScatteringTransform(
            in_channels=in_channels,
            J=J,
            num_moments=num_moments,
        )
        self.scattering.eval()

    def __call__(self, data):
        """Apply scattering transform to a single graph."""
        with torch.no_grad():
            data.batch = torch.zeros(data.x.size(0), dtype=torch.long)
            coeffs = self.scattering(data)
        return coeffs.squeeze(0).detach(), data.y.squeeze(0)

    def out_shape(self):
        return self.scattering.out_shape()

In [10]:
# ============================================================
# CONFIGURATION - Update this path to your experiment directory
# ============================================================
experiment_dir = 'outputs/MOSES_12K_fixed_regress_nokld_2026-01-22-13-24-10'

# Load saved config
config_path = f'{experiment_dir}/config.yaml'
with open(config_path, 'r') as f:
    config = yaml.safe_load(f)

print("Loaded config from:", config_path)
print(f"  GRASSY version: {config['training']['grassy_version']}")
print(f"  Scattering J={config['scattering']['J']}, moments={config['scattering']['num_moments']}")

# Extract config sections
dataset_cfg = config['dataset']
model_cfg = config['model']
training_cfg = config['training']
scattering_cfg = config['scattering']

FileNotFoundError: [Errno 2] No such file or directory: 'outputs/MOSES_12K_fixed_regress_nokld_2026-01-22-13-24-10/config.yaml'

In [ ]:
# Load base dataset (without transform)
base_dataset = ZINCDataset(
    dataset_cfg['path'], 
    prop_stat_dict=dataset_cfg.get('stats_path'),
    transform=None
)

# Create fixed scattering transform with config parameters
scattering_transform = FixedScatteringTransform(
    in_channels=base_dataset.num_node_features,
    J=scattering_cfg['J'],
    num_moments=scattering_cfg['num_moments']
)

# Pre-compute scattering coefficients for the dataset
print("\nPre-computing scattering coefficients...")
scattering_data = []
for data in tqdm(base_dataset, desc="Computing scattering"):
    coeffs, props = scattering_transform(data)
    scattering_data.append((coeffs, props))

In [7]:
# Determine model filename from config
grassy_version = training_cfg['grassy_version']
kl_div = grassy_version in ['VAE', 'VAE+REG']
reg = grassy_version in ['AE+REG', 'VAE+REG']
reg_str = 'regress' if reg else 'noregress'
kl_str = 'kld' if kl_div else 'nokld'
prefix = f"{dataset_cfg['name']}_{reg_str}_{kl_str}"

# Load model
model_path = f'{experiment_dir}/{prefix}_model.pt'
print(f"\nLoading model from: {model_path}")


Loading model from: outputs/MOSES_12K_fixed_regress_nokld_2026-01-22-13-24-10/MOSES_12K_regress_nokld_model.pt


In [ ]:
model_state = torch.load(model_path, map_location='cpu')

In [ ]:
# Get sample dimensions
sample_x, sample_y = scattering_data[0]

# Property names (adjust to match your dataset)
prop_names = ['qed', 'HeavyAtomMolWt', 'MolWt', 'BalabanJ', 'BertzCT', 'Ipc', 
              'TPSA', 'NumHAcceptors', 'NumHDonors', 'RingCount', 'MolLogP', 'SAscore', 'FSP3']

# Determine alpha and beta based on GRASSY version
alpha = training_cfg['alpha'] if reg else 0
beta = training_cfg['beta'] if kl_div else 0

# Create model with config hyperparameters
hparams = Namespace(
    input_dim=len(sample_x), 
    bottle_dim=model_cfg['bottle_dim'],
    hidden_dim=model_cfg['hidden_dim'],
    learning_rate=training_cfg['learning_rate'], 
    alpha=alpha,
    beta=beta,
    n_epochs=training_cfg['n_epochs'],
    len_epoch=None, 
    batch_size=training_cfg['batch_size'], 
    n_gpus=config['hardware']['n_gpus'], 
    num_properties=len(sample_y)
)

print(f"\nModel config:")
print(f"  input_dim: {hparams.input_dim}")
print(f"  bottle_dim: {hparams.bottle_dim}")
print(f"  hidden_dim: {hparams.hidden_dim}")
print(f"  alpha: {hparams.alpha}, beta: {hparams.beta}")

model = GRASSY(hparams=hparams)
model.load_state_dict(model_state)
model.eval()

# Compute test indices based on config splits
train_size = dataset_cfg['train_size']
val_size = dataset_cfg['val_size']
test_start = train_size + val_size
test_indices = list(range(test_start, len(scattering_data)))

print(f"\nDataset splits from config:")
print(f"  Train: 0-{train_size-1} ({train_size} samples)")
print(f"  Val: {train_size}-{train_size+val_size-1} ({val_size} samples)")
print(f"  Test: {test_start}-{len(scattering_data)-1} ({len(test_indices)} samples)")
print("=" * 70)

# Store all errors and true values per property
all_abs_errors = []
all_true_values = []

with torch.no_grad():
    for idx in tqdm(test_indices, desc="Evaluating"):
        x, y = scattering_data[idx]
        x_t = x.unsqueeze(0).float()
        y_t = y.unsqueeze(0).float() if torch.is_tensor(y) else torch.tensor(y).unsqueeze(0).float()
        
        _, y_hat, _, _, _ = model(x_t)
        
        # Absolute error per property
        abs_error = torch.abs(y_hat - y_t).squeeze(0)
        all_abs_errors.append(abs_error.numpy())
        all_true_values.append(y_t.squeeze(0).numpy())

# Convert to arrays: [N_samples, N_properties]
all_abs_errors = np.array(all_abs_errors)
all_true_values = np.array(all_true_values)

print("\n" + "=" * 70)
print("PROPERTY PREDICTION ACCURACY (Per Property)")
print("=" * 70)
print(f"{'Property':<20} | {'MAE':<12} | {'Std Dev':<12} | {'Range (min-max)':<20}")
print("-" * 70)

# Calculate and print stats per property
for i in range(all_abs_errors.shape[1]):
    prop_mae = all_abs_errors[:, i].mean()
    prop_std = all_abs_errors[:, i].std()
    prop_min = all_true_values[:, i].min()
    prop_max = all_true_values[:, i].max()
    
    # Use property name if available
    p_name = prop_names[i] if i < len(prop_names) else f"Prop {i+1}"
    
    print(f"{p_name:<20} | {prop_mae:<12.6f} | {prop_std:<12.6f} | [{prop_min:.3f}, {prop_max:.3f}]")

print("=" * 70)
print("\nOverall (averaged across all properties):")
print(f"  Mean Absolute Error: {all_abs_errors.mean():.6f}")
print(f"  Standard Deviation: ±{all_abs_errors.std():.6f}")